# Blihat: SuperPoint + LightGlue-only baseline

Runs the fine-stage-only tracker (`--tracker keypoint`, no DINOv3) on UAV123, DTB70 and UAV20L, and compares it with your existing Blihat and coarse-only results.

1. **Setup**: GPU check, Google Drive for results, clone and install the repo.
2. **Datasets**: download from Kaggle with `kagglehub` and arrange them in the layout the got10k toolkit expects.
3. **Tuning sweep**: tune the baseline on the UAV123 scale subset (24 sequences, 300 frames), the same protocol that selected Blihat's frozen config.
4. **Full benchmarks**: the baseline with Blihat's frozen settings (`keypoint_final`) and with the tuned settings (`keypoint_tuned`), on all three benchmarks.
5. **Results**: the comparison table and paired statistics (mean AUC gain, bootstrap 95% CI, paired t), the same numbers `RESULTS.txt` reports for coarse vs. Blihat.

**Before you start**

- Use a GPU runtime (*Runtime → Change runtime type*). Your Blihat numbers were measured on an A100; use the same GPU type so FPS can be compared.
- Push the latest commit (keypoint tracker + sweep) to GitHub first. This notebook clones the repo.
- Add your Kaggle credentials as Colab secrets: 🔑 in the left sidebar → `KAGGLE_USERNAME` and `KAGGLE_KEY` (from kaggle.com → Settings → API → *Create New Token*), with *Notebook access* on.
- Fill in the Kaggle dataset names in **step 2**.
- To compare with Blihat, upload your existing result folders to Drive (see **step 5**). No DINOv3 or Hugging Face login is needed: this notebook never runs Blihat itself.

Results go to Google Drive, and finished sequences are skipped. If Colab disconnects, re-run the cells from the top and the runs continue where they stopped.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Where results are stored. On Drive they survive a disconnect and runs resume.
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/blihat'
else:
    WORK = '/content/blihat_work'
RUNS = f'{WORK}/runs'                 # new results
REFERENCE = f'{WORK}/reference'       # your existing Blihat / coarse-only results (step 5)
os.makedirs(RUNS, exist_ok=True)
print('results ->', RUNS)

In [ ]:
REPO_URL = 'https://github.com/DagimawiMoges/Zero-shot-UAV-object-tracking-with-coarse-to-fine-architecture.git'
REPO = '/content/Blihat'

if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git log -1 --format='commit %h  %s'
assert os.path.exists('blihat/keypoint_tracker.py') and os.path.exists('eval/sweep_keypoint.py'), \
    'The cloned repo has no keypoint tracker yet: push your latest commit to GitHub and re-run this cell.'

!pip install -q -r requirements.txt kagglehub
!pip install -q git+https://github.com/cvg/LightGlue.git
!pip install -q -e .

## 2. Datasets from Kaggle

Set the Kaggle dataset names (`owner/dataset`, as in the dataset's URL `kaggle.com/datasets/owner/dataset`).

- `UAV123` must contain the frames (`data_seq/UAV123/...`) and the annotations (`anno/UAV123`). Most uploads also include `anno/UAV20L`.
- `UAV20L`: leave it as `None` if the UAV123 upload already has `anno/UAV20L`. UAV20L uses the UAV123 frames, so only its annotations are needed.
- `DTB70` must contain the 70 sequence folders (`<seq>/img/*.jpg` + `groundtruth_rect.txt`).

The next cell searches the downloads for these folders, so the upload's own folder structure doesn't matter.

In [ ]:
KAGGLE_DATASETS = {
    'UAV123': 'OWNER/UAV123-DATASET',   # <- fill in
    'DTB70':  'OWNER/DTB70-DATASET',    # <- fill in
    'UAV20L': None,                     # only if anno/UAV20L is not inside the UAV123 upload
}

for k, v in KAGGLE_DATASETS.items():
    if v and v.startswith('OWNER/'):
        raise ValueError(f"Set KAGGLE_DATASETS['{k}'] to the Kaggle dataset name (owner/dataset).")

In [ ]:
# Kaggle credentials from Colab secrets (or ~/.kaggle/kaggle.json, or an interactive login)
os.environ.setdefault('KAGGLEHUB_CACHE', '/content/kaggle_cache')   # local disk, not Drive
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials: Colab secrets')
except Exception as e:
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print(f'No Colab secrets ({type(e).__name__}); asking for a login.')
        import kagglehub
        kagglehub.login()

import kagglehub
DOWNLOADS = {}
for name, slug in KAGGLE_DATASETS.items():
    if slug:
        print(f'Downloading {name}: {slug} (can take a while the first time) ...')
        DOWNLOADS[name] = kagglehub.dataset_download(slug)
        print('  ->', DOWNLOADS[name])

In [ ]:
import glob, json, os, zipfile
import got10k

META = json.load(open(os.path.join(os.path.dirname(got10k.__file__), 'datasets', 'uav123.json')))
UAV_FOLDERS = {m['folder_name'] for v in META.values() for m in v.values()}


def unzip_nested(root):
    """Extract any .zip files that are still inside a download (some uploads nest them)."""
    for z in glob.glob(os.path.join(root, '**', '*.zip'), recursive=True):
        out = z[:-4]
        if not os.path.isdir(out):
            print('  extracting', os.path.relpath(z, root))
            with zipfile.ZipFile(z) as f:
                f.extractall(out)


def _walk(roots, max_depth=6):
    for root in roots:
        root = os.path.realpath(root)
        base = root.count(os.sep)
        for d, subdirs, files in os.walk(root, followlinks=True):
            if d.count(os.sep) - base >= max_depth:
                subdirs[:] = []
            subdirs[:] = [s for s in subdirs if not s.startswith('.') and s != '__MACOSX']
            yield d, subdirs, files


def find_uav_frames(roots):
    """Folder whose subfolders are UAV123 sequence folders (bike1/000001.jpg, ...)."""
    best, score = None, 0
    for d, subdirs, _ in _walk(roots):
        hits = [s for s in subdirs if s in UAV_FOLDERS]
        if len(hits) > score and os.path.exists(os.path.join(d, hits[0], '000001.jpg')):
            best, score = d, len(hits)
    return best, score


def find_anno(roots, version):
    """Folder with one <sequence>.txt per sequence of `version` (UAV123 or UAV20L)."""
    names = set(META[version])
    best, score = None, 0
    for d, _, files in _walk(roots):
        n = len(names & {f[:-4] for f in files if f.endswith('.txt')})
        if n > score:
            best, score = d, n
    return (best, score) if score >= 0.9 * len(names) else (None, score)


def find_dtb70(roots):
    """Folder whose subfolders are DTB70 sequences (<seq>/img/*.jpg + groundtruth_rect.txt)."""
    best, score = None, 0
    for d, subdirs, _ in _walk(roots):
        n = sum(os.path.exists(os.path.join(d, s, 'groundtruth_rect.txt'))
                and os.path.isdir(os.path.join(d, s, 'img')) for s in subdirs)
        if n > score:
            best, score = d, n
    return best, score


def _link(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst):
        os.remove(dst)
    if not os.path.exists(dst):
        os.symlink(src, dst)


def build_uav_root(frames, anno123, anno20l, out):
    """got10k layout: <out>/data_seq/UAV123/<folder>/000001.jpg, <out>/anno/{UAV123,UAV20L}/*.txt."""
    _link(frames, os.path.join(out, 'data_seq', 'UAV123'))
    if anno123:
        _link(anno123, os.path.join(out, 'anno', 'UAV123'))
    if anno20l:
        _link(anno20l, os.path.join(out, 'anno', 'UAV20L'))
    # got10k's integrity check wants more than 3 entries in the root folder
    for f in ('LAYOUT.txt', 'SOURCE.txt'):
        with open(os.path.join(out, f), 'w') as fh:
            fh.write('Symlinked got10k layout built by the Blihat benchmark notebook.\n')
    return out


def build_flat_anno(dtb_root, out):
    """<out>/<seq>.txt -> <dtb_root>/<seq>/groundtruth_rect.txt (for eval/analyze_uav123.py)."""
    os.makedirs(out, exist_ok=True)
    for g in glob.glob(os.path.join(dtb_root, '*', 'groundtruth_rect.txt')):
        _link(g, os.path.join(out, os.path.basename(os.path.dirname(g)) + '.txt'))
    return out


# ---- find the folders and build got10k layouts (symlinks, no copying) ----
DATA = '/content/data'
for p in DOWNLOADS.values():
    unzip_nested(p)
uav_roots = [DOWNLOADS['UAV123']] + ([DOWNLOADS['UAV20L']] if 'UAV20L' in DOWNLOADS else [])

frames, n_fr = find_uav_frames(uav_roots)
anno123, n123 = find_anno(uav_roots, 'UAV123')
anno20l, n20 = find_anno(uav_roots, 'UAV20L')
dtb, n_dtb = find_dtb70([DOWNLOADS['DTB70']])
print(f'UAV123 frames : {frames}  ({n_fr} sequence folders)')
print(f'UAV123 anno   : {anno123}  ({n123}/123 files)')
print(f'UAV20L anno   : {anno20l}  ({n20}/20 files)')
print(f'DTB70         : {dtb}  ({n_dtb} sequences)')
assert frames and anno123, 'UAV123 frames or annotations not found in the download: check the dataset name.'
assert dtb and n_dtb >= 70, 'DTB70 sequences not found (or incomplete) in the download: check the dataset name.'
if not anno20l:
    print("!! UAV20L annotations not found: UAV20L is skipped. Set KAGGLE_DATASETS['UAV20L'] to a dataset that has them.")

UAV_ROOT = build_uav_root(frames, anno123, anno20l, f'{DATA}/UAV123')
DTB_ROOT = dtb
DTB_ANNO = build_flat_anno(dtb, f'{DATA}/DTB70_anno')
ANNO = {'UAV123': f'{UAV_ROOT}/anno/UAV123', 'UAV20L': f'{UAV_ROOT}/anno/UAV20L', 'DTB70': DTB_ANNO}
ROOT = {'UAV123': UAV_ROOT, 'UAV20L': UAV_ROOT, 'DTB70': DTB_ROOT}
BENCHMARKS = ['UAV123', 'DTB70'] + (['UAV20L'] if anno20l else [])

# Fail fast: every sequence must have its first/last frame and matching annotation counts
# (got10k would otherwise stop mid-run, hours in)
import contextlib, io
from got10k.datasets import UAV123 as _U, DTB70 as _D
with contextlib.redirect_stdout(io.StringIO()):
    checks = {'UAV123': _U(UAV_ROOT, 'UAV123'), 'DTB70': _D(DTB_ROOT)}
    if anno20l:
        checks['UAV20L'] = _U(UAV_ROOT, 'UAV20L')
bad = {}
for name, ds in checks.items():
    for seq in ds.seq_names:
        try:
            imgs, anno = ds[seq]
            ok = len(imgs) > 0 and os.path.exists(imgs[0]) and os.path.exists(imgs[-1])
        except Exception:
            ok = False
        if not ok:
            bad.setdefault(name, []).append(seq)
    n_bad = len(bad.get(name, []))
    print(f"{'OK ' if not n_bad else '!! '}{name}: {len(ds.seq_names) - n_bad}/{len(ds.seq_names)} sequences complete")
if bad:
    raise RuntimeError('Incomplete dataset: ' + '; '.join(f'{k}: {", ".join(v[:10])}{" ..." if len(v) > 10 else ""}'
                                                         for k, v in bad.items()))

## 3. Tuning sweep (UAV123 scale subset)

The same protocol that selected Blihat's config: 24 sequences, first 300 frames. It runs the frozen settings, nine single changes, then the combination of the changes that helped. Expect a few hours. If Colab disconnects, re-run from the top; finished variants are skipped.

In [ ]:
!python eval/sweep_keypoint.py --dataset-root {ROOT['UAV123']} --anno {ANNO['UAV123']} --result-dir {RUNS}/kp_sweep

In [ ]:
import json
sweep = json.load(open(f'{RUNS}/kp_sweep/keypoint_sweep.json'))
best = next(v for v in sweep['variants'] if v['name'] == sweep['best'])
TUNED_SETS = best['overrides']
print('best variant:', best['name'], f"(AUC {best['auc']:.3f} on the subset)")
print('tuned overrides:', TUNED_SETS or 'none: the frozen config is already best, keypoint_tuned is skipped')

## 4. Full benchmarks

Two runs per benchmark: `keypoint_final` (Blihat's frozen settings) and `keypoint_tuned` (best sweep variant). Run the cells one at a time if you want to split the work across sessions.

In [ ]:
def run(benchmark, name, sets=()):
    sets = ' '.join(f'--set {s}' for s in sets)
    get_ipython().system(
        f"python eval/run_benchmark.py --tracker keypoint --config configs/blihat_final.yaml {sets} "
        f"--dataset {benchmark} --dataset-root {ROOT[benchmark]} "
        f"--result-dir {RUNS}/full --name {name}")

def run_both(benchmark):
    if benchmark not in BENCHMARKS:
        print(f'{benchmark}: skipped (data not available)'); return
    run(benchmark, 'keypoint_final')
    if TUNED_SETS:
        run(benchmark, 'keypoint_tuned', TUNED_SETS)

In [ ]:
run_both('UAV123')

In [ ]:
run_both('DTB70')

In [ ]:
run_both('UAV20L')

## 5. Results

To compare with Blihat and the coarse-only baseline, upload your existing result folders (the ones with one `<sequence>.txt` per sequence) to Drive:

```
MyDrive/blihat/reference/UAV123/icaad_final      MyDrive/blihat/reference/UAV123/coarse_final
MyDrive/blihat/reference/DTB70/icaad_final_dtb   MyDrive/blihat/reference/DTB70/coarse_final_dtb
MyDrive/blihat/reference/UAV20L/icaad_final_uav20l   MyDrive/blihat/reference/UAV20L/coarse_final
```

These are the folder names on your computer. Edit `REF` below if you rename them. Missing folders are simply left out of the table.

In [ ]:
import sys
import numpy as np
sys.path.insert(0, 'eval')
from analyze_uav123 import evaluate

REF = {
    'UAV123': {'Blihat': 'UAV123/icaad_final',        'DINOv3 only': 'UAV123/coarse_final'},
    'DTB70':  {'Blihat': 'DTB70/icaad_final_dtb',     'DINOv3 only': 'DTB70/coarse_final_dtb'},
    'UAV20L': {'Blihat': 'UAV20L/icaad_final_uav20l', 'DINOv3 only': 'UAV20L/coarse_final'},
}

def paired(a, b, rng=np.random.default_rng(0)):
    # Per-sequence AUC gain of a over b: mean, bootstrap 95% CI, paired t, better/worse counts
    common = sorted(set(a) & set(b))
    d = np.array([a[s]['auc'] - b[s]['auc'] for s in common])
    boot = rng.choice(d, (10000, len(d))).mean(1)
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 and d.std() > 0 else float('nan')
    return dict(n=len(d), gain=d.mean(), lo=np.percentile(boot, 2.5), hi=np.percentile(boot, 97.5), t=t,
                better=int((d > 0.005).sum()), worse=int((d < -0.005).sum()))

summary = {}
for bm in BENCHMARKS:
    runs = {}
    for label, rel in REF[bm].items():
        if os.path.isdir(f'{REFERENCE}/{rel}'):
            runs[label] = evaluate(f'{REFERENCE}/{rel}', ANNO[bm])
    for name, label in (('keypoint_final', 'LightGlue only (frozen cfg)'),
                        ('keypoint_tuned', 'LightGlue only (tuned)')):
        if os.path.isdir(f'{RUNS}/full/{bm}/{name}'):
            runs[label] = evaluate(f'{RUNS}/full/{bm}/{name}', ANNO[bm])
    runs = {k: v for k, v in runs.items() if v}
    if not runs:
        continue
    print(f'\n{bm}')
    print(f"  {'tracker':30s} {'seqs':>4s} {'AUC':>6s} {'Prec':>6s} {'FPS':>6s}")
    for label, rows in runs.items():
        auc = np.mean([r['auc'] for r in rows.values()]); prec = np.mean([r['prec'] for r in rows.values()])
        fps = np.nanmedian([r['fps'] for r in rows.values()])
        print(f'  {label:30s} {len(rows):4d} {auc:6.3f} {prec:6.3f} {fps:6.1f}')
        summary.setdefault(bm, {})[label] = dict(seqs=len(rows), auc=float(auc), prec=float(prec), fps=float(fps))
    if 'Blihat' in runs:
        for label in [k for k in runs if k.startswith('LightGlue')]:
            p = paired(runs['Blihat'], runs[label])
            print(f"  Blihat vs {label}: gain {p['gain']:+.3f}, 95% CI [{p['lo']:+.3f}, {p['hi']:+.3f}], "
                  f"t = {p['t']:.1f}, better on {p['better']}/{p['n']}, worse on {p['worse']}")
            summary[bm][f'Blihat vs {label}'] = {k: float(v) for k, v in p.items()}

with open(f'{RUNS}/keypoint_baseline_summary.json', 'w') as f:
    json.dump(summary, f, indent=1)
print(f'\nSaved {RUNS}/keypoint_baseline_summary.json')

Per-sequence detail for one run (largest drops/gains vs. Blihat, by class and attribute):

In [ ]:
bm = 'UAV123'
!python eval/analyze_uav123.py --results {RUNS}/full/{bm}/keypoint_final --anno {ANNO[bm]} --baseline {REFERENCE}/{REF[bm]['Blihat']} --top 10